# 5. Photometry & Lightcurves

In this tutorial, you will learn how to run the Photometry Pipeline to extract time-series light curves for detected stars, which can be used to identify variable stars or exoplanet transits.


## 5.1 Running the Photometry Pipeline

We will use `astrometrics.processing.process_target(target, stages=["photometry"])` to extract light curves from individual unstacked light frames, using a solved reference frame for alignment.


In [1]:
from astrometricslib import Astrometrics, FilterType

astrometrics = Astrometrics()
target = astrometrics.targets.get("M 13")

print("Starting photometry pipeline...")
result = astrometrics.processing.process_target(
    target, stages=["photometry"], photometry={"filter_name": FilterType.L, "use_astrometry_seed": True}
).results["photometry"]
print(f"Photometry Pipeline Status: {result.get('status', 'completed')}")
print(f"Stars found: {result.get('starsFound', 0)}")

Starting photometry pipeline...


The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


INFO: Query finished. [astroquery.utils.tap.core]


Photometry Pipeline Status: completed
Stars found: 482


## 5.2 Variability Analysis

A "variable star" is one whose brightness changes over time rather than just appearing to vary because of measurement noise. The pipeline identifies candidates using a composite three-index assessment against the field noise model:

1. **Excess Scatter:** The ratio of a star's sample standard deviation to the noise level expected for its brightness.
2. **Reduced Chi-Square ($\chi_\nu^2$):** Goodness-of-fit of the normalized light curve against a constant flux assumption.
3. **Stetson $J$ Index:** Evaluates whether consecutive residual measurements correlate, acting as a veto against isolated noise spikes.

When a field has too few stars to fit a full noise model, the pipeline falls back to flagging unusually high coefficients of variation ($C_v$) relative to the field median.

:::{note}
**Time Standard.** Every point on an extracted light curve is timestamped in **Barycentric Julian Date in Barycentric Dynamical Time ($\text{BJD}_{\text{TDB}}$)**, correcting for Earth's orbital light-travel time (up to $\pm 8.3$ minutes) to prevent phase smearing in periodic and transit folding.
:::

In [2]:
variable_candidates = result.get("variableCandidates", [])
print(f"Variable star candidates found: {len(variable_candidates)}")

for candidate in variable_candidates:
    print(
        f"  {candidate.id}: mean flux={candidate.mean_flux:.4f}, Cv={candidate.coefficient_of_variation:.4f}"
    )

Variable star candidates found: 7
  HD 150998: mean flux=15.6108, Cv=0.0417
  FIELD_J249.8349+35.5320: mean flux=8.5440, Cv=0.0395
  HD 151086: mean flux=6.1277, Cv=0.0254
  BD+36  2775: mean flux=5.9421, Cv=0.0258
  NGC  6205   169: mean flux=0.1803, Cv=0.0404
  NGC  6205   353: mean flux=0.1826, Cv=0.0594
  NGC  6205   176: mean flux=0.1229, Cv=0.0570
